# 01 — Data Cleaning & Dataset Preparation

**Project:** Classical Pneumothorax Detection using NIH ChestX-ray14

This notebook prepares a balanced subset of 500 X-ray images: 250 Pneumothorax and 250 No Finding.

## 1. Setup

**Local Kaggle API setup:** this notebook expects a Kaggle API token at `~/.kaggle/kaggle.json` (Kaggle account -> *Create New API Token*, then move the downloaded file there). Never commit `kaggle.json` to the repo.

In [1]:
import pandas as pd

DATASET = "achmadbauravindah/nih-chest-xrays-original"
CSV_PATH = "../data/Data_Entry_2017.csv"
SUBSET_PATH = "../data/pneumothorax_500_metadata.csv"


## 2. Download and load metadata

In [ ]:
!mkdir -p ../data
!kaggle datasets download -d achmadbauravindah/nih-chest-xrays-original -f Data_Entry_2017.csv -p ../data

df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)


## 3. Select classes

In [3]:
pneumothorax_df = df[df["Finding Labels"] == "Pneumothorax"].copy()
no_finding_df = df[df["Finding Labels"] == "No Finding"].copy()

print("Pneumothorax:", len(pneumothorax_df))
print("No Finding:", len(no_finding_df))

Pneumothorax: 2194
No Finding: 60361


## 4. Create balanced 500-image subset

In [4]:
subset_df = pd.concat([
    pneumothorax_df.sample(n=250, random_state=42),
    no_finding_df.sample(n=250, random_state=42)
], ignore_index=True)

subset_df["Label"] = (
    subset_df["Finding Labels"] == "Pneumothorax"
).astype(int)

print("Class distribution:")
print(subset_df["Label"].value_counts())
print("Total images:", len(subset_df))

Class distribution:
Label
1    250
0    250
Name: count, dtype: int64
Total images: 500


## 5. Save cleaned metadata

In [ ]:
subset_df.to_csv(SUBSET_PATH, index=False)

print("Saved:", SUBSET_PATH)
print("Columns:", list(subset_df.columns))
print("First 5 images:")
display(subset_df[["Image Index", "Finding Labels", "Label"]].head())

In [ ]:
import os

os.makedirs("../data/images", exist_ok=True)

print("Image folder ready:", os.path.exists("../data/images"))


In [7]:
import os
import subprocess
import re

target_files = set(subset_df["Image Index"])
found = {}

page_token = None
page = 1

while len(found) < len(target_files):
    cmd = [
        "kaggle", "datasets", "files", DATASET,
        "--page-size", "200"
    ]

    if page_token:
        cmd += ["--page-token", page_token]

    result = subprocess.run(cmd, capture_output=True, text=True)
    text = result.stdout

    paths = re.findall(r"(images_\d+/images/[^\s]+\.png)", text)

    for path in paths:
        filename = os.path.basename(path)
        if filename in target_files:
            found[filename] = path

    match = re.search(r"Next Page Token = (.+)", text)

    print(f"Page {page}: found {len(found)}/{len(target_files)} images")

    if not match:
        break

    page_token = match.group(1).strip()
    page += 1

print("\nFinished searching.")
print("Images found:", len(found))

Page 1: found 1/500 images
Page 2: found 1/500 images
Page 3: found 1/500 images
Page 4: found 1/500 images
Page 5: found 1/500 images
Page 6: found 2/500 images
Page 7: found 2/500 images
Page 8: found 5/500 images
Page 9: found 5/500 images
Page 10: found 7/500 images
Page 11: found 7/500 images
Page 12: found 8/500 images
Page 13: found 10/500 images
Page 14: found 10/500 images
Page 15: found 11/500 images
Page 16: found 12/500 images
Page 17: found 12/500 images
Page 18: found 13/500 images
Page 19: found 15/500 images
Page 20: found 15/500 images
Page 21: found 15/500 images
Page 22: found 16/500 images
Page 23: found 17/500 images
Page 24: found 17/500 images
Page 25: found 17/500 images
Page 26: found 19/500 images
Page 27: found 20/500 images
Page 28: found 22/500 images
Page 29: found 22/500 images
Page 30: found 24/500 images
Page 31: found 24/500 images
Page 32: found 24/500 images
Page 33: found 25/500 images
Page 34: found 26/500 images
Page 35: found 27/500 images
Page 3

In [ ]:
for i, path in enumerate(found.values(), 1):
    subprocess.run([
        "kaggle", "datasets", "download",
        "-d", DATASET,
        "-f", path,
        "-p", "../data/images"
    ], stdout=subprocess.DEVNULL)

    if i % 25 == 0:
        print(f"Downloaded {i}/500 images")

print("Download complete!")


In [ ]:
import os

downloaded_files = [
    f for f in os.listdir("../data/images")
    if f.endswith(".png")
]

print("Images downloaded:", len(downloaded_files))
print("Images expected:", len(subset_df))


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os

sample_file = downloaded_files[0]
sample_path = os.path.join("../data/images", sample_file)

img = Image.open(sample_path)

print("Image:", sample_file)
print("Size:", img.size)
print("Mode:", img.mode)

plt.imshow(img, cmap="gray")
plt.axis("off")
plt.show()
